# Milestone 7 — Multimodal Fusion

## 7.1 — Fusion Dataset & Pipeline

### Objective

The objective of this phase was to construct the dataset and embedding extraction pipeline required for multimodal fusion using the representations learned during self-supervised pretraining.

Unlike the previous self-supervised learning stage, this phase uses only **complete multimodal visits**. Each sample must contain all three modalities:

```text
Image + Radiograph + Text

Missing-modality scenarios are intentionally excluded from this phase. Robustness to missing modalities will be evaluated separately in Milestone 8 — Missing-Modality Robustness.

Dataset

The authoritative patient-level split was preserved.

No new patient-level split was created.

The dataset used for fusion is:

results/labeled_patient_level_dataset/labeled_dataset.csv

The same six diagnostic labels used throughout the supervised baseline experiments were retained:

label_caries
label_gingivitis
label_malocclusion
label_pulpitis
label_tooth_loss
label_tooth_structure_loss

The six-label representation is kept consistent with the existing downstream classification experiments so that the fusion results remain directly comparable with the previous baselines.

Patient-Level Split

The existing patient-level split was reused without modification.

The authoritative split contains:

Train:
    3,360 patients
    6,129 visits

Validation:
    720 patients
    1,330 visits

Test:
    720 patients
    1,316 visits

No patient appears in more than one split.

Patient-level leakage:
    PASS

Complete Multimodal Filtering

For multimodal fusion, a visit is included only when all three required modalities are available:

Image
+
Radiograph
+
Clinical Text

Visits with one or more missing modalities are excluded from the fusion dataset.

This filtering is specific to the fusion training stage and does not modify the original dataset or patient-level split.

The resulting fusion dataset sizes are:

Split	Complete Multimodal Visits
Train	2,935
Validation	627
Test	633
Total	4,195

The resulting distribution is consistent with the previously observed number of complete multimodal visits in the COde dataset.

Fusion Sample Structure

Each fusion sample contains:

checkup_id
patient_id

images
radiographs

input_ids
attention_mask

labels

The image and radiograph fields contain all available images associated with the visit.

Because the number of images can vary between visits, the dataset preserves the image collections rather than forcing a fixed number of images at the dataset level.

Modality-Specific Image Resolution

The COde dataset stores filenames in the CSV rather than complete filesystem paths.

For example:

0001-001-01.jpg

The same filename may exist in both modalities:

Images/Photographs/0001-001-01.jpg
Images/Radiographs/0001-001-01.jpg

Therefore, image resolution is explicitly modality-aware.

The fusion preprocessing pipeline resolves paths as:

Image
    → Images/Photographs/

Radiograph
    → Images/Radiographs/

This prevents accidental cross-modality file resolution when identical filenames exist in both directories.

Image Preprocessing

The same image preprocessing strategy used during SSL training was retained:

Resize
    → 224 × 224

ToTensor

ImageNet normalization

Normalization:

mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]

Multiple images belonging to a single visit are processed independently.

Before being passed to the corresponding SSL encoder, they are mean-pooled at the image level:

Image 1 ─┐
Image 2 ─┤
Image 3 ─┼── Mean Pooling ── SSL Image Encoder
   ...   ┤
Image N ─┘

The same strategy is applied to radiographs.

Text Preprocessing

Clinical text is constructed from the same clinical fields used by the downstream text baseline:

chief_complaint
present_illness
past_medical_record
examination

The following fields are not used as input because they may directly or indirectly reveal diagnostic labels:

anomalies_en
diagnosis
treatment_plan
management

The resulting clinical text is tokenized using the same DistilBERT tokenizer used during SSL pretraining.

Maximum sequence length:

256 tokens

The resulting inputs are:

input_ids
attention_mask
SSL Encoder Representations

The pretrained multimodal SSL model is used only as a representation extractor.

The SSL projection heads are not used for fusion.

The extraction pipeline is:

Photographs
    ↓
Image preprocessing
    ↓
Mean pooling
    ↓
SSL Image Encoder
    ↓
2048-dimensional embedding
Radiographs
    ↓
Radiograph preprocessing
    ↓
Mean pooling
    ↓
SSL Radiograph Encoder
    ↓
2048-dimensional embedding
Clinical Text
    ↓
DistilBERT tokenizer
    ↓
SSL Text Encoder
    ↓
768-dimensional embedding

The resulting representation for each visit is therefore:

Image Embedding       → 2048-d
Radiograph Embedding  → 2048-d
Text Embedding        → 768-d

The total raw concatenated representation size is:

2048 + 2048 + 768 = 4864 dimensions
SSL Checkpoint

The pretrained SSL representation model was loaded from:

results/ssl_pretraining/full_dynamic/best_ssl_model.pt

The encoder weights were loaded from the completed multimodal SSL pretraining stage.

The projection heads used for the contrastive objective were intentionally excluded from the fusion representations because they are optimized for the SSL contrastive objective rather than downstream diagnosis.

Fusion DataLoader

A dedicated multimodal collate function was implemented because different visits contain different numbers of images and radiographs.

The DataLoader therefore preserves the per-visit image collections while batching:

Text tensors
Labels
Patient IDs
Checkup IDs

The extraction pipeline was successfully executed with:

Batch size: 4
Device: CUDA
Embedding Extraction Results

The embedding extraction pipeline was successfully completed for all three patient-level splits.

Split	Samples	Image Dim	Radiograph Dim	Text Dim
Train	2,935	2048	2048	768
Validation	627	2048	2048	768
Test	633	2048	2048	768

All extracted representations passed dimensionality validation.

Output Artifacts

The extracted SSL representations were saved as:

results/ssl_pretraining/fusion_embeddings/
├── train.pt
├── validation.pt
└── test.pt

Each stored sample contains:

checkup_id
patient_id
image_embedding
radiograph_embedding
text_embedding
labels
Implementation

The fusion-specific implementation was organized under:

src/ssl/fusion/

The main components include:

src/ssl/fusion/
├── config.py
├── dataset.py
├── preprocess.py
├── extract_embeddings.py
└── test_dataset.py

This keeps the multimodal fusion implementation separate from the general SSL training code and from the supervised baseline implementation.

Validation

The dataset pipeline was tested successfully.

The sample-level validation produced:

checkup_id        → str
patient_id        → str

images            → torch.Size([1, 3, 224, 224])
radiographs       → torch.Size([1, 3, 224, 224])

input_ids         → torch.Size([256])
attention_mask    → torch.Size([256])

labels            → torch.Size([6])

The successful test confirms that:

all three modalities are present for fusion samples;
image and radiograph paths are resolved correctly;
modality-specific path resolution is preserved;
clinical text is tokenized correctly;
six diagnostic labels are returned;
patient-level splits are preserved;
SSL encoder representations can be extracted successfully.
Observation

The complete multimodal fusion dataset contains:

2,935 train visits
627 validation visits
633 test visits

for a total of:

4,195 complete multimodal visits

This represents the subset of the dataset where Image, Radiograph, and Text are simultaneously available.

The resulting representations provide a fixed-size multimodal input for the next phase:

Image       → 2048
Radiograph  → 2048
Text        → 768
                 ↓
              Fusion
                 ↓
          6-label diagnosis

No missing-modality experiments are performed in this phase.

Conclusion

Phase 7.1 successfully established the complete multimodal fusion data pipeline.

The authoritative patient-level split and six-label classification protocol were preserved. Complete multimodal visits were isolated without creating a new split, and the pretrained SSL encoders successfully generated fixed-dimensional representations for Image, Radiograph, and Text.

The resulting embeddings are now ready to be used by the fusion models.

Phase 7.1 Status
[x] Patient-level split preserved
[x] Six-label classification protocol preserved
[x] Complete multimodal filtering
[x] MultimodalFusionDataset
[x] Modality-aware image path resolution
[x] Image preprocessing
[x] Radiograph preprocessing
[x] Text preprocessing
[x] Fusion DataLoader
[x] SSL checkpoint loading
[x] Image embedding extraction
[x] Radiograph embedding extraction
[x] Text embedding extraction
[x] Train embedding extraction
[x] Validation embedding extraction
[x] Test embedding extraction
[x] Embedding dimension validation

Phase 7.1 — COMPLETE
Next Step

The next phase is:

Phase 7.2 — Baseline Fusion

The first fusion model will use a simple early-fusion strategy:

Image Embedding
       +
Radiograph Embedding
       +
Text Embedding
       ↓
Concatenation
       ↓
MLP Classifier
       ↓
6 Diagnostic Labels

This model will provide the simple fusion baseline against which the main fusion architecture will later be compared.